# 10 - Phase 2 Analysis

Compute final success tables, convergence estimates, bootstrap confidence intervals, effect sizes, and the Phase 1 vs Phase 2 comparison inputs.

In [9]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np

from src.utils.io import DIRS
from src.utils.metrics_phase2 import bootstrap_ci, cohens_d

## Load Phase 2 Results

In [10]:
def parse_run_id(run_id):
    parts = run_id.split("_")
    return {
        "mode": parts[1],
        "algo": parts[2],
        "reward": parts[3],
        "level": parts[4],
        "difficulty": parts[5],
        "seed": int(parts[6].replace("seed", "")),
    }

def load_runs(mode="full"):
    runs = []
    for path in sorted(DIRS["metrics"].glob("p2_*_metrics.npz")):
        run_id = path.name.replace("_metrics.npz", "")
        meta = parse_run_id(run_id)
        if mode is not None and meta["mode"] != mode:
            continue
        data = np.load(path)
        runs.append({"run_id": run_id, **meta, **{k: data[k] for k in data.files}})
    return runs

MODE = "quick"  # use "quick" for the current saved DDQN run; change to "full" after full training
runs = load_runs(mode=MODE)
print("loaded", len(runs), "runs in mode", MODE)
if not runs:
    print("No runs found for this mode. Available modes:")
    modes = sorted({parse_run_id(p.name.replace("_metrics.npz", ""))["mode"] for p in DIRS["metrics"].glob("p2_*_metrics.npz")})
    print(modes)

loaded 5 runs in mode quick


## Metric Functions

In [11]:
def final_rate(arr, window=100):
    arr = np.asarray(arr, dtype=float)
    if len(arr) == 0:
        return np.nan
    return float(np.mean(arr[-min(window, len(arr)):]))

def convergence_episode(success, window=100, threshold=0.90):
    success = np.asarray(success, dtype=float)
    if len(success) < window:
        return None
    kernel = np.ones(window) / window
    rolling = np.convolve(success, kernel, mode="valid")
    hits = np.where(rolling >= threshold)[0]
    return int(hits[0] + window) if len(hits) else None

def summarise_run(run):
    success = run["success"]
    successes = run["steps"][success == 1]
    return {
        "algo": run["algo"],
        "reward": run["reward"],
        "seed": run["seed"],
        "success_rate": final_rate(run["success"]),
        "key_rate": final_rate(run["key"]),
        "door_rate": final_rate(run["door"]),
        "mean_return": float(np.mean(run["rewards"][-min(100, len(run["rewards"])):])),
        "mean_steps_success": float(np.mean(successes)) if len(successes) else np.nan,
        "convergence_episode": convergence_episode(run["success"]),
    }

run_summaries = [summarise_run(run) for run in runs]
for row in run_summaries:
    print(row)

print("\nCurrent analysis coverage:")
print("  runs:", len(run_summaries))
print("  algorithms:", sorted({r["algo"] for r in run_summaries}))
print("  rewards:", sorted({r["reward"] for r in run_summaries}))
print("  seeds:", sorted({r["seed"] for r in run_summaries}))

{'algo': 'ddqn', 'reward': 'potential', 'seed': 0, 'success_rate': 0.0, 'key_rate': 0.25, 'door_rate': 0.03, 'mean_return': 17.77650000000015, 'mean_steps_success': 304.46153846153845, 'convergence_episode': None}
{'algo': 'ddqn', 'reward': 'sparse', 'seed': 0, 'success_rate': 0.29, 'key_rate': 0.72, 'door_rate': 0.42, 'mean_return': -9.159399999999964, 'mean_steps_success': 147.78155339805826, 'convergence_episode': None}
{'algo': 'ddqn', 'reward': 'subgoal', 'seed': 0, 'success_rate': 0.77, 'key_rate': 0.99, 'door_rate': 0.91, 'mean_return': -0.7962999999999871, 'mean_steps_success': 88.91743119266054, 'convergence_episode': None}
{'algo': 'dqn', 'reward': 'subgoal', 'seed': 0, 'success_rate': 0.93, 'key_rate': 0.94, 'door_rate': 0.94, 'mean_return': -0.12689999999999355, 'mean_steps_success': 102.49850746268656, 'convergence_episode': 514}
{'algo': 'ppo', 'reward': 'subgoal', 'seed': 0, 'success_rate': 0.0, 'key_rate': 0.0, 'door_rate': 0.0, 'mean_return': -3.9999999999999587, 'mean

## 3x3 Algorithm x Reward Table

In [12]:
def group_values(rows, algo, reward, key):
    return np.array([r[key] for r in rows if r["algo"] == algo and r["reward"] == reward], dtype=float)

ALGOS = ["dqn", "ddqn", "ppo"]
REWARDS = ["sparse", "subgoal", "potential"]

table = []
printed_effect = False
for algo in ALGOS:
    for reward in REWARDS:
        vals = group_values(run_summaries, algo, reward, "success_rate")
        if len(vals) == 0:
            continue
        ci_low, ci_high = bootstrap_ci(vals, seed=42)
        table.append({
            "algo": algo,
            "reward": reward,
            "n": len(vals),
            "success_mean": float(np.mean(vals)),
            "success_std": float(np.std(vals)),
            "ci_low": ci_low,
            "ci_high": ci_high,
        })

if table:
    for row in table:
        print(row)
else:
    print("No aggregate table rows. Run at least one Phase 2 metrics file for this MODE.")

{'algo': 'dqn', 'reward': 'subgoal', 'n': 1, 'success_mean': 0.93, 'success_std': 0.0, 'ci_low': 0.93, 'ci_high': 0.93}
{'algo': 'ddqn', 'reward': 'sparse', 'n': 1, 'success_mean': 0.29, 'success_std': 0.0, 'ci_low': 0.29, 'ci_high': 0.29}
{'algo': 'ddqn', 'reward': 'subgoal', 'n': 1, 'success_mean': 0.77, 'success_std': 0.0, 'ci_low': 0.77, 'ci_high': 0.77}
{'algo': 'ddqn', 'reward': 'potential', 'n': 1, 'success_mean': 0.0, 'success_std': 0.0, 'ci_low': 0.0, 'ci_high': 0.0}
{'algo': 'ppo', 'reward': 'subgoal', 'n': 1, 'success_mean': 0.0, 'success_std': 0.0, 'ci_low': 0.0, 'ci_high': 0.0}


## Reward Ablation Effect Sizes

In [13]:
for algo in ALGOS:
    sparse = group_values(run_summaries, algo, "sparse", "success_rate")
    subgoal = group_values(run_summaries, algo, "subgoal", "success_rate")
    potential = group_values(run_summaries, algo, "potential", "success_rate")
    if len(sparse) and len(subgoal):
        printed_effect = True
        print(algo, "subgoal vs sparse d=", cohens_d(subgoal, sparse))
    if len(sparse) and len(potential):
        printed_effect = True
        print(algo, "potential vs sparse d=", cohens_d(potential, sparse))

if not printed_effect:
    print("No effect sizes yet: need at least sparse + subgoal or sparse + potential for the same algorithm.")
    print("Current data has only:", sorted({(r["algo"], r["reward"]) for r in run_summaries}))

ddqn subgoal vs sparse d= nan
ddqn potential vs sparse d= nan


## Phase 1 Vs Phase 2 Input Check

Use this section after Phase 1 and Phase 2 sparse DQN runs are saved. It prints the files to compare for V10.

In [14]:
phase1_candidates = sorted(DIRS["metrics"].glob("dqn_*_metrics.npz"))
phase2_candidates = sorted(DIRS["metrics"].glob(f"p2_{MODE}_dqn_sparse_*_metrics.npz"))
print("Phase 1 DQN candidates:")
for p in phase1_candidates:
    print(" ", p.name)
print("Phase 2 DQN sparse candidates:")
for p in phase2_candidates:
    print(" ", p.name)

Phase 1 DQN candidates:
Phase 2 DQN sparse candidates:
